# 1. What is a Decision Tree?

## Concept
A **Decision Tree** is a supervised machine learning algorithm used for both classification and regression tasks. It learns to partition data based on feature values, forming a tree-like model of decisions.

**Why is it called a tree?**
Because the model structure resembles an upside-down tree. It starts at a single root node and branches out into possible outcomes, finally ending in "leaves" which represent the predictions.

**Basic Intuition**
Think of a Decision Tree as a game of "20 Questions". The model asks a series of yes/no questions about the data features to narrow down the possibilities until it reaches a conclusion.

**Classification vs Regression Trees**
* **Classification Tree:** Predicts a categorical label (discrete). E.g., Will the customer buy this product? (Yes/No)
* **Regression Tree:** Predicts a continuous numerical value. E.g., What will be the price of this house? ($)

### Real-world Applications
* **Loan Approval:** Should a bank approve a loan based on income and credit score?
* **Disease Classification:** Diagnosing an illness based on patient symptoms.
* **Customer Churn:** Predicting whether a customer will cancel their subscription.
* **House Price Prediction:** Estimating the value of a property based on its features (regression).

### The Decision Process
Data → Question → Decision → Question → Final Prediction

> **Key Idea:** A Decision Tree breaks down a complex decision-making process into a collection of simpler, sequential decisions.


# 2. Structure of a Decision Tree

## Concept
A Decision Tree is made up of several types of nodes and connections:

* **Root Node:** The very top node. It represents the entire dataset before any splits are made.
* **Internal Node:** A node where a question is asked (a feature is evaluated), splitting the data further.
* **Branch:** The connection between nodes, representing the outcome of a decision/test (e.g., True or False).
* **Leaf Node (Terminal Node):** The final nodes at the bottom. They do not split further and contain the final prediction (class label or numerical value).
* **Parent Node:** A node that splits into sub-nodes.
* **Child Node:** The sub-nodes created when a parent node splits.
* **Depth:** The length of the longest path from the root node to a leaf node.

### Simple Visual Diagram
```mermaid
graph TD
    A[Root Node: Age > 30?] -->|Yes| B[Internal Node: Income > 50k?]
    A -->|No| C[Leaf Node: Reject]
    B -->|Yes| D[Leaf Node: Approve]
    B -->|No| E[Leaf Node: Reject]
```

**Example:**
* **Root Node:** We check `Age > 30?` on all applicants.
* **Branch:** The paths for "Yes" and "No".
* **Internal Node:** For those over 30, we ask another question: `Income > 50000?` (Parent to the final leaves).
* **Leaf Node:** The final decisions: "Approve" or "Reject".


# 3. How a Decision Tree Makes Predictions

## Concept
Making a prediction (inference) with a trained Decision Tree is straightforward and highly interpretable.

**Step-by-step process:**
1. **Start at root:** Take a new data instance and feed it to the root node.
2. **Check condition:** Evaluate the feature condition at the node (e.g., Is Age > 30?).
3. **Follow appropriate branch:** Move down the left or right branch based on the answer (True or False).
4. **Repeat:** Continue checking conditions at internal nodes and moving down the branches.
5. **Reach leaf:** The process stops when you arrive at a leaf node.
6. **Make prediction:** The prediction is the majority class (for classification) or average value (for regression) of the training samples in that leaf node.

### Example Decision Path
Let's predict loan approval for a new applicant: **Age = 35, Income = 60,000**.

1. Start at Root: `Age > 30?` -> 35 > 30 is **True**. Move to the left child node.
2. Internal Node: `Income > 50000?` -> 60,000 > 50,000 is **True**. Move to the left child node.
3. Leaf Node: Reached the leaf node. Prediction is **Approve**.

> **Remember:** A prediction is just tracing a single path from the root to a leaf based on the feature values of a new sample.


# 4. Splitting Data

## Concept
The core of training a Decision Tree is figuring out how to partition the data.

* **What is a split?** A split is a rule (e.g., `feature_value <= threshold`) that divides a dataset into two subsets.
* **Why do we split data?** To create groups that are more homogeneous (pure) with respect to the target variable than the original group.
* **How does a tree choose the best split?** It evaluates all possible splits across all features and chooses the one that results in the most "pure" child nodes.

### Purity and Impurity
* **Purity:** A node is pure if all samples in it belong to the same class. (e.g., a node with 10 "Approve" and 0 "Reject" samples).
* **Impurity:** A node is impure if it contains a mix of classes. (e.g., 5 "Approve" and 5 "Reject" samples).

> **Key Idea:** The algorithm's goal at every step is to minimize impurity (maximize purity) in the resulting child nodes. It wants to separate the classes as quickly and cleanly as possible.


# 5. Entropy

## Concept
**Entropy** is a mathematical measure of impurity or randomness in a dataset. It comes from Information Theory.

### Formula
$$ H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i) $$

Where:
* $S$ is the dataset (or subset in a node).
* $c$ is the number of classes.
* $p_i$ is the probability (proportion) of class $i$ in the node.

### Understanding Entropy
* **Pure Node:** If a node has only one class, $p_1 = 1$, and $\log_2(1) = 0$. So, Entropy = 0 (No impurity).
* **Impure Node:** If a node has an equal mix of 2 classes (50/50), Entropy = 1 (Maximum impurity for binary classification).

Let's calculate this manually using Python.


In [1]:
import numpy as np

def entropy(class_counts):
    """Calculates entropy of a node given the counts of each class."""
    total = sum(class_counts)
    if total == 0:
        return 0
    
    ent = 0
    for count in class_counts:
        if count > 0:
            p_i = count / total
            ent -= p_i * np.log2(p_i)
            
    return ent

# Example 1: Pure Node (10 Apples, 0 Oranges)
pure_node = [10, 0]
print(f"Entropy of Pure Node (10, 0): {entropy(pure_node):.4f}")

# Example 2: Completely Impure Node (5 Apples, 5 Oranges)
impure_node = [5, 5]
print(f"Entropy of Impure Node (5, 5): {entropy(impure_node):.4f}")

# Example 3: Somewhat Impure Node (8 Apples, 2 Oranges)
mixed_node = [8, 2]
print(f"Entropy of Mixed Node (8, 2): {entropy(mixed_node):.4f}")


Entropy of Pure Node (10, 0): 0.0000
Entropy of Impure Node (5, 5): 1.0000
Entropy of Mixed Node (8, 2): 0.7219


# 6. Information Gain

## Concept
**Information Gain (IG)** is the metric used to decide which feature to split on at each step. It measures how much the entropy (impurity) decreases after a split.

### Formula
$$ \text{Information Gain} = \text{Entropy(Parent)} - \text{Weighted Average Entropy(Children)} $$

### Why is it useful?
* The tree evaluates all possible splits.
* For each split, it calculates the Information Gain.
* It chooses the split that yields the **highest Information Gain** (i.e., the split that reduces entropy the most).

Let's calculate Information Gain manually using Python.


In [2]:
def information_gain(parent_counts, left_counts, right_counts):
    """Calculates information gain for a split."""
    # Parent Entropy
    parent_ent = entropy(parent_counts)
    
    # Total samples
    total_samples = sum(parent_counts)
    
    # Weighted Entropy of Children
    weight_left = sum(left_counts) / total_samples
    weight_right = sum(right_counts) / total_samples
    
    child_ent = (weight_left * entropy(left_counts)) + (weight_right * entropy(right_counts))
    
    # Information Gain
    ig = parent_ent - child_ent
    return ig

# Scenario:
# Parent Node: 10 Positive, 10 Negative (Entropy = 1.0)
parent = [10, 10]

# Split Option A gives: Left(8 Pos, 2 Neg), Right(2 Pos, 8 Neg)
left_A = [8, 2]
right_A = [2, 8]
ig_A = information_gain(parent, left_A, right_A)

# Split Option B gives: Left(6 Pos, 4 Neg), Right(4 Pos, 6 Neg)
left_B = [6, 4]
right_B = [4, 6]
ig_B = information_gain(parent, left_B, right_B)

print(f"Information Gain for Split A: {ig_A:.4f}")
print(f"Information Gain for Split B: {ig_B:.4f}")
print("\nConclusion: Split A provides more information gain and is the better choice.")


Information Gain for Split A: 0.2781
Information Gain for Split B: 0.0290

Conclusion: Split A provides more information gain and is the better choice.


# 7. Gini Impurity

## Concept
**Gini Impurity** is an alternative to Entropy for measuring node impurity. It measures the probability of incorrectly classifying a randomly chosen element if it were randomly labeled according to the distribution of classes in the node.

### Formula
$$ \text{Gini} = 1 - \sum_{i=1}^{c} p_i^2 $$

Where $p_i$ is the probability of an item belonging to class $i$.

### Understanding Gini
* **Pure Node:** If a node is 100% one class, Gini = 1 - (1^2) = 0.
* **Mixed Node:** For a 50/50 split in binary classification, Gini = 1 - (0.5^2 + 0.5^2) = 1 - (0.25 + 0.25) = 0.5.
* Gini impurity is bounded between 0 and 0.5 (for binary classification), whereas Entropy is bounded between 0 and 1.

Let's calculate Gini impurity in Python.


In [3]:
def gini_impurity(class_counts):
    """Calculates Gini impurity of a node."""
    total = sum(class_counts)
    if total == 0:
        return 0
    
    gini = 1.0
    for count in class_counts:
        p_i = count / total
        gini -= p_i ** 2
        
    return gini

print(f"Gini of Pure Node (10, 0): {gini_impurity([10, 0]):.4f}")
print(f"Gini of Impure Node (5, 5): {gini_impurity([5, 5]):.4f}")
print(f"Gini of Mixed Node (8, 2): {gini_impurity([8, 2]):.4f}")


Gini of Pure Node (10, 0): 0.0000
Gini of Impure Node (5, 5): 0.5000
Gini of Mixed Node (8, 2): 0.3200


# 8. Entropy vs Gini

## Concept
Both Entropy and Gini Impurity are used to evaluate the quality of a split. They behave very similarly and usually result in the same tree structure, but there are slight differences.

| Feature | Entropy | Gini Impurity |
| :--- | :--- | :--- |
| **Formula** | $H = -\sum p_i \log_2(p_i)$ | $Gini = 1 - \sum p_i^2$ |
| **Range (Binary)** | 0 to 1 | 0 to 0.5 |
| **Interpretation**| Measures information/randomness | Measures misclassification probability |
| **Speed** | Slightly slower (logarithmic calculation) | Slightly faster (squaring calculation) |
| **Common Use** | Default in C4.5 algorithm | Default in Scikit-learn (CART algorithm) |

> **Remember:** In practice, the choice between Gini and Entropy has little impact on the final model's performance. Gini is often preferred simply because it is computationally faster to calculate.


# 9. Decision Tree from Scratch — Simple Version

## Concept
Let's build a tiny, manual decision tree step-by-step on a very small dataset to solidify how splitting works. We won't build a full recursive class, just demonstrate evaluating splits.


In [4]:
import pandas as pd

# Tiny dataset: Predict if someone will play tennis based on Weather and Temp
data = {
    'Weather': ['Sunny', 'Sunny', 'Overcast', 'Rainy', 'Rainy'],
    'Temp': [85, 80, 83, 70, 68],
    'Play': ['No', 'No', 'Yes', 'Yes', 'No']
}
df = pd.DataFrame(data)
print("Dataset:\n", df)

# Helper for Gini
def get_gini(series):
    counts = series.value_counts()
    total = len(series)
    return 1 - sum((count/total)**2 for count in counts)

# Current Impurity (Root)
root_gini = get_gini(df['Play'])
print(f"\nRoot Gini Impurity: {root_gini:.4f}")

# Let's evaluate splitting by Weather == 'Sunny'
left_mask = df['Weather'] == 'Sunny'
left_split = df[left_mask]['Play']
right_split = df[~left_mask]['Play']

gini_left = get_gini(left_split)
gini_right = get_gini(right_split)

# Weighted Gini
weight_l = len(left_split) / len(df)
weight_r = len(right_split) / len(df)
weighted_gini = (weight_l * gini_left) + (weight_r * gini_right)

print(f"\nSplit 1: Weather == 'Sunny'")
print(f"Left node (Sunny) Gini: {gini_left:.4f} (Samples: {len(left_split)})")
print(f"Right node (Not Sunny) Gini: {gini_right:.4f} (Samples: {len(right_split)})")
print(f"Weighted Average Gini: {weighted_gini:.4f}")
print(f"Impurity Decrease (Gain): {root_gini - weighted_gini:.4f}")

# In a real algorithm, it repeats this for ALL features and ALL possible thresholds, 
# then picks the split with the lowest Weighted Average Gini (highest gain).


Dataset:
     Weather  Temp Play
0     Sunny    85   No
1     Sunny    80   No
2  Overcast    83  Yes
3     Rainy    70  Yes
4     Rainy    68   No

Root Gini Impurity: 0.4800

Split 1: Weather == 'Sunny'
Left node (Sunny) Gini: 0.0000 (Samples: 2)
Right node (Not Sunny) Gini: 0.4444 (Samples: 3)
Weighted Average Gini: 0.2667
Impurity Decrease (Gain): 0.2133


# 10. Decision Tree with Scikit-learn

## Concept
In practice, we use `Scikit-learn` to efficiently train Decision Trees. We will use the `DecisionTreeClassifier`.

We will use the breast cancer dataset built into sklearn.


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Load dataset
data = load_breast_cancer()

# 2. Separate X and y
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Create model
# Using criterion="gini" (default) and random_state=42 for reproducibility
model = DecisionTreeClassifier(criterion="gini", random_state=42)

# 5. Fit model
model.fit(X_train, y_train)

# 6. Predict
y_pred = model.predict(X_test)

# 7. Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Decision Tree Accuracy: {accuracy:.4f}")


# 11. Visualizing a Decision Tree

## Concept
One of the biggest advantages of Decision Trees is that they are highly interpretable. We can visualize the exact decision path the model learned.

We use `plot_tree` from `sklearn.tree`.

### Understanding the Plot
* **First line:** The feature and threshold used for the split (e.g., `mean concave points <= 0.049`).
* **gini/entropy:** The impurity of the node.
* **samples:** The number of observations in this node.
* **value:** The count of samples in each class.
* **class:** The majority class predicted by this node.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# We will train a smaller tree just for visualization so it fits on screen
small_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
small_tree.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(
    small_tree, 
    feature_names=data.feature_names.tolist(),  
    class_names=data.target_names.tolist(),
    filled=True, 
    rounded=True, 
    fontsize=10
)
plt.title("Decision Tree Visualization (max_depth=3)")
plt.show()


# 12. Decision Tree Hyperparameters

## Concept
Hyperparameters control the growth and complexity of the tree. Tuning them is critical to prevent overfitting.

Important `DecisionTreeClassifier` parameters:

* `criterion`: The function to measure the quality of a split (`'gini'` or `'entropy'`).
* `max_depth`: The maximum depth of the tree. Limits how many levels the tree can grow. Very useful for controlling overfitting.
* `min_samples_split`: The minimum number of samples required to split an internal node. Higher values prevent the tree from splitting on very small datasets.
* `min_samples_leaf`: The minimum number of samples required to be at a leaf node. Prevents leaves with only 1 or 2 samples.
* `max_features`: The number of features to consider when looking for the best split.
* `max_leaf_nodes`: Grow a tree with `max_leaf_nodes` in best-first fashion based on impurity reduction.

> **Important:** Unconstrained trees (default sklearn settings) will grow until all leaves are pure, often leading to massive overfitting.


# 13. Overfitting in Decision Trees

## Concept
Decision trees are extremely prone to **overfitting**. If left unconstrained, a tree will keep splitting until every leaf contains exactly one sample, perfectly memorizing the training data. However, this complex tree will generalize poorly to new, unseen data.

Let's demonstrate this by varying `max_depth`.


In [ ]:
train_scores = []
test_scores = []
depths = range(1, 15)

for depth in depths:
    # Train tree of specific depth
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    
    # Record accuracies
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

# Plotting the results
plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, label='Training Accuracy', marker='o')
plt.plot(depths, test_scores, label='Testing Accuracy', marker='s')
plt.xlabel('Tree Depth (max_depth)')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Overfitting as Depth Increases')
plt.axvline(x=3, color='gray', linestyle='--', label='Optimal Depth (approx)')
plt.legend()
plt.grid(True)
plt.show()


### Explanation
* **Underfitting (Left side):** At depth 1 or 2, the tree is too simple to capture the patterns in the data. Both train and test accuracy are lower.
* **Good Fit (Middle):** Around depth 3-4, the test accuracy peaks. The model learns generalizable patterns.
* **Overfitting (Right side):** As depth increases beyond 4, training accuracy approaches 1.0 (100%), but testing accuracy drops or plateaus. The model is memorizing noise in the training set.


# 14. Pruning

## Concept
**Pruning** is the process of reducing the size of a Decision Tree to prevent overfitting and improve generalization. It essentially "cuts off" branches that add little predictive power.

### Types of Pruning
1. **Pre-pruning (Early Stopping):** Stopping the tree from growing before it perfectly classifies the training set.
    * Controlled via hyperparameters: `max_depth`, `min_samples_split`, `min_samples_leaf`.
    * *This is the most common approach used in Scikit-learn.*
2. **Post-pruning (Cost-Complexity Pruning):** Growing the full tree first, and then working backward from the leaves to remove splits that don't provide sufficient improvement.
    * Scikit-learn supports this via the `ccp_alpha` parameter. Increasing `ccp_alpha` increases the penalty for having a complex tree, forcing pruning.

> **Key Idea:** Pruning simplifies the model. A simpler model is less likely to overfit the training data.


# 15. Decision Tree for Regression

## Concept
Decision Trees can also predict continuous numerical values.

* **Classification Tree:** Leaf nodes contain the *majority class*. Splits use Gini/Entropy.
* **Regression Tree:** Leaf nodes contain the *average (mean)* of the target values for samples in that node. Splits usually use **Mean Squared Error (MSE)** to evaluate impurity. The goal is to find splits that minimize the variance of the target variable within the child nodes.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate synthetic sine wave data
rng = np.random.RandomState(42)
X_reg = np.sort(5 * rng.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel()
y_reg[::5] += 3 * (0.5 - rng.rand(16)) # Add noise

# Create and fit the model
reg_tree = DecisionTreeRegressor(max_depth=4, random_state=42)
reg_tree.fit(X_reg, y_reg)

# Predict
y_reg_pred = reg_tree.predict(X_reg)

# Evaluate
print(f"MAE: {mean_absolute_error(y_reg, y_reg_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_reg, y_reg_pred)):.4f}")
print(f"R² Score: {r2_score(y_reg, y_reg_pred):.4f}")

# Plotting the regression tree prediction line
X_test_reg = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_test_pred = reg_tree.predict(X_test_reg)

plt.figure(figsize=(10, 6))
plt.scatter(X_reg, y_reg, s=20, edgecolor="black", c="darkorange", label="Data")
plt.plot(X_test_reg, y_test_pred, color="cornflowerblue", label="max_depth=4", linewidth=2)
plt.xlabel("X")
plt.ylabel("Target")
plt.title("Decision Tree Regression")
plt.legend()
plt.show()


# 16. Feature Scaling and Decision Trees

## Concept
**Decision Trees DO NOT require feature scaling** (like Standardization or Normalization).

### Why?
A Decision Tree makes splits based on logical conditions for one feature at a time (e.g., `Income > 50000`). The splitting algorithm simply sorts the values of a single feature and finds the optimal threshold.

Scaling `Income` from `[10000, 100000]` to `[0, 1]` doesn't change the order of the values. The tree will just find a new threshold (e.g., `Scaled_Income > 0.5`) that perfectly corresponds to the old one. The resulting tree structure and predictions will be identical.

**Comparison:**
* **KNN / Logistic Regression:** Rely on distance metrics or gradient descent. They **require** feature scaling, otherwise features with larger scales dominate.
* **Decision Trees (and Random Forests):** Scale-invariant. They **do not require** feature scaling.

> **Important:** Skipping the scaling step saves time and keeps the model's feature thresholds easily interpretable in their original units.


# 17. Missing Values and Categorical Data

## Concept

### Missing Values
The mathematical theory behind Decision Trees allows them to handle missing values (e.g., by sending data down both branches proportionally).
* **However, Scikit-learn's implementation currently does NOT support missing values natively.**
* **Solution:** You must impute missing values (e.g., using `SimpleImputer` to fill with median/mode) before passing data to a Scikit-learn Decision Tree.

### Categorical Data
Decision Trees can split on categorical features (e.g., `Color == 'Red'`).
* **However, Scikit-learn's CART implementation requires all features to be numerical.**
* **Solution:** You must encode categorical features using `OneHotEncoder` (for nominal data) or `OrdinalEncoder` (for ordinal data) before training.


# 18. Feature Importance

## Concept
Once a tree is trained, you can ask it which features were the most useful for making decisions.

* Decision Trees calculate **Feature Importance** based on how much each feature decreases impurity (Gini/Entropy) across all the nodes where that feature is used for a split. Features that cause large reductions in impurity closer to the root node get higher importance scores.

Let's extract and plot feature importance from our classification model.


In [ ]:
# Assuming 'model' is our trained DecisionTreeClassifier on the breast cancer dataset
importances = model.feature_importances_

# Create a DataFrame
feature_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
})

# Sort features by importance
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
plt.barh(feature_imp_df['Feature'][:10][::-1], feature_imp_df['Importance'][:10][::-1], color='teal')
plt.xlabel('Gini Importance')
plt.title('Top 10 Feature Importances in Decision Tree')
plt.show()


> **Common Mistake:** A feature importance of 0 does not mean the feature is completely useless in reality; it just means the specific tree didn't select it for a split. Also, feature importances in highly correlated features can be somewhat arbitrary.


# 19. Advantages of Decision Trees

* **Easy to understand:** Highly interpretable. You can literally trace the decision logic.
* **Easy to visualize:** The tree structure can be drawn out.
* **Little preprocessing:** Requires no feature scaling (standardization/normalization).
* **Handles nonlinear relationships:** Can easily model complex, non-linear boundaries.
* **Works for classification and regression:** Extremely versatile.
* **Can model feature interactions:** Naturally captures interactions between variables.


# 20. Disadvantages of Decision Trees

* **Can overfit:** Without proper pruning, they build complex models that memorize noise.
* **Sensitive to small data changes:** A tiny change in training data can cause a completely different tree (high variance).
* **Deep trees can become complex:** They lose interpretability if they grow too large.
* **Greedy splitting does not guarantee globally optimal trees:** It chooses the best split locally at each node.
* **Feature importance can be misleading:** For highly correlated features, it arbitrarily assigns high importance to one and zero to the other.
* **Single trees may have lower stability than ensembles:** This is why Random Forests are generally preferred in production.


# 21. Decision Tree vs Other Algorithms

| Feature | Decision Tree | KNN | Logistic Regression |
| :--- | :--- | :--- | :--- |
| **Interpretability** | Very High | Low | High |
| **Scaling requirement** | **Not Required** | Required | Required |
| **Training speed** | Fast | Instant (Lazy) | Fast |
| **Prediction speed**| Very Fast | Slow | Very Fast |
| **Nonlinear relationships** | Excellent | Good | Poor (without transforms) |
| **Overfitting tendency**| Very High | High (Low K) | Moderate |


# 22. Complete End-to-End Decision Tree Project

## Concept
We will apply everything we learned to build a complete classification pipeline using the classic Iris dataset.


In [ ]:
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# 1. Import libraries & 2. Load dataset
iris = load_iris()

# 3. Convert to DataFrame
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target
df_iris['species'] = df_iris['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# 4. Inspect data
display(df_iris.head())

# 5. Perform basic EDA (Pairplot to see feature separation)
sns.pairplot(df_iris.drop('target', axis=1), hue='species', palette='Dark2')
plt.suptitle("Iris Dataset Pairplot", y=1.02)
plt.show()

# 6. Separate X and y
X_iris = df_iris.drop(['target', 'species'], axis=1)
y_iris = df_iris['target']

# 7. Train-test split
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)

# 8. Create & 9. Train model
model_pruned = DecisionTreeClassifier(max_depth=3, random_state=42)
model_pruned.fit(X_train_i, y_train_i)

# 10. Predict
y_pred_i = model_pruned.predict(X_test_i)

# 11-14. Evaluate Metrics
print("--- Model Evaluation (max_depth=3) ---")
print(f"Accuracy:  {accuracy_score(y_test_i, y_pred_i):.4f}")
print(f"Precision: {precision_score(y_test_i, y_pred_i, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test_i, y_pred_i, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test_i, y_pred_i, average='weighted'):.4f}")

# 15. Confusion matrix
cm = confusion_matrix(y_test_i, y_pred_i)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# 16. Plot decision tree
plt.figure(figsize=(12, 8))
plot_tree(model_pruned, feature_names=iris.feature_names, class_names=iris.target_names.tolist(), filled=True, rounded=True)
plt.title("Pruned Decision Tree (max_depth=3)")
plt.show()

# 17. Plot feature importance
importances_i = model_pruned.feature_importances_
plt.figure(figsize=(8, 4))
sns.barplot(x=importances_i, y=iris.feature_names, palette='viridis', hue=iris.feature_names, legend=False)
plt.title("Feature Importances")
plt.xlabel("Importance")
plt.show()


### Explain the results
* The EDA pairplot showed that petal length and petal width are highly effective at separating the species.
* The feature importance plot confirms that the tree relied entirely on petal dimensions.
* Limiting `max_depth=3` created a simple, highly interpretable tree that generalizes perfectly to the test set without overfitting.


# 23. Common Decision Tree Mistakes

1. **Creating an unrestricted deep tree**
   * **Problem:** Overfitting. The model memorizes training data.
   * **Better approach:** Always use pruning parameters like `max_depth` or `min_samples_split`.

2. **Ignoring overfitting**
   * **Problem:** Assuming high training accuracy means a great model.
   * **Better approach:** Evaluate on a separate test set. Compare train vs. test scores.

3. **Choosing hyperparameters randomly**
   * **Problem:** Guessing `max_depth` leads to suboptimal models.
   * **Better approach:** Use grid search to systematically find the best parameters.

4. **Evaluating only on training data**
   * **Problem:** Leads to a false sense of security.
   * **Better approach:** Strict Train/Test split workflow.

5. **Ignoring class imbalance**
   * **Problem:** The tree will bias towards predicting the majority class.
   * **Better approach:** Use class weights or balance the dataset before training.

6. **Misinterpreting feature importance**
   * **Problem:** Thinking a feature with 0 importance is useless everywhere.
   * **Better approach:** Realize feature importance is model-specific and affected by correlated features.

7. **Assuming deeper means better**
   * **Problem:** Unnecessary complexity that hurts test accuracy.
   * **Better approach:** Start shallow and increase depth only if validation metrics improve.

8. **Using unnecessary preprocessing**
   * **Problem:** Applying StandardScaler to data before training a tree.
   * **Better approach:** Save processing time; trees don't need scaling.

9. **Data leakage**
   * **Problem:** Including a feature that is effectively the target variable in disguise.
   * **Better approach:** Rigorously review features.

10. **Not using validation/cross-validation**
    * **Problem:** Decision trees have high variance. A single split might give a lucky evaluation.
    * **Better approach:** Use cross-validation to get a stable estimate.


# 24. Interview Questions

1. **What is a Decision Tree?** A supervised algorithm that recursively partitions data based on feature thresholds to make predictions.
2. **What is a Root Node?** The top node containing the entire dataset before any splits.
3. **What is a Leaf Node?** A terminal node that makes the final prediction.
4. **What is Entropy?** A measure of randomness or impurity in a node.
5. **What is Gini Impurity?** The probability of incorrectly classifying a randomly chosen element in the node.
6. **What is Information Gain?** The reduction in impurity achieved by splitting a node.
7. **How is the best split selected?** The algorithm evaluates possible splits and selects the one that maximizes Information Gain.
8. **Gini vs Entropy?** Both measure impurity. Gini is computationally faster. They yield similar trees.
9. **Why do trees overfit?** They will continue splitting until leaves are perfectly pure, memorizing noise.
10. **What is pruning?** Limiting tree growth or removing branches to reduce complexity and prevent overfitting.
11. **What is max_depth?** A hyperparameter that sets the maximum number of levels a tree can have.
12. **What is min_samples_split?** Minimum samples a node must have to be allowed to split further.
13. **What is min_samples_leaf?** Minimum samples required to be present in a newly created leaf node.
14. **Does a Decision Tree require feature scaling?** No. Splitting is based on sorting values.
15. **What is feature importance?** A score indicating how much a feature contributed to reducing impurity.
16. **Decision Tree classification vs regression?** Classification predicts classes using majority voting. Regression predicts continuous values using mean.
17. **Advantages of Decision Trees?** Interpretable, no scaling, handles non-linear data well.
18. **Disadvantages of Decision Trees?** Prone to overfitting, high variance/unstable.
19. **Why are trees unstable?** A small change in data might change the root split, changing the entire tree.
20. **What is greedy splitting?** Making the best optimal split at the current node without looking ahead.
21. **How do you prevent overfitting?** Pruning parameters (`max_depth`) or using ensembles.
22. **Can they handle categorical variables?** Conceptually yes, but Scikit-learn requires numerical encoding.
23. **What is Cost-Complexity Pruning?** A post-pruning technique that penalizes the tree based on complexity.
24. **How does a tree handle imbalanced data?** Badly. It biases toward the majority class.
25. **When to use Logistic Regression over a Decision Tree?** When the relationship is strictly linear.


# 25. Quick Revision Cheat Sheet

| Concept | Definition |
| :--- | :--- |
| **Root** | The first node containing all data. |
| **Node** | Point where a condition is evaluated. |
| **Branch** | Outcome path of a split. |
| **Leaf** | Terminal node outputting prediction. |
| **Entropy** | Measure of impurity/randomness. |
| **Information Gain**| Amount of entropy removed by a split. |
| **Gini Impurity** | Measure of misclassification probability. |
| **Splitting** | Dividing a node based on a condition. |
| **Max Depth** | Limit on how deep the tree can grow. |
| **Min Samples Split**| Minimum samples needed to split a node. |
| **Min Samples Leaf** | Minimum samples allowed in a leaf node. |
| **Pruning** | Restricting tree size to prevent overfitting. |
| **Feature Importance**| Ranks features by impurity reduction. |
| **Overfitting** | Memorizes noise, fails on new data. |
| **Underfitting** | Fails to capture data patterns. |
| **Classification Tree**| Predicts discrete classes. |
| **Regression Tree** | Predicts continuous numbers. |

### Decision Tree Workflow
Data → Choose Best Split → Create Branches → Repeat Splitting → Stop Condition → Leaf → Prediction → Pruning / Tuning → Evaluation


# 26. Practice Problems

1. Calculate entropy manually.
2. Calculate Gini impurity.
3. Calculate information gain.
4. Train a Decision Tree classifier.
5. Visualize the tree.
6. Compare different max_depth values.
7. Plot feature importance.
8. Build a Decision Tree regressor.
9. Compare Gini and Entropy.
10. Apply pruning/tuning and evaluate performance.

---

## Next Notebook
`07_random_forest.ipynb`

In the next notebook, we will cover Random Forest and ensemble learning to solve the stability and overfitting problems of single Decision Trees!
